# Volumetric Surface reconstruction

In todays exercise, we will reconstruct a surface from a point cloud obtained by a structured light scan.

In [ ]:
from pygel3d import hmesh, jupyter_display as jd
jd.set_export_mode(True)
import numpy as np
from numpy.linalg import norm
from math import exp,floor,sqrt
from random import shuffle
array = np.array

### Part 1: Preliminaries

A few tools are required and therefore defined below

In [ ]:
def Gaussian(p0,p,ssq):
    """ Compute the value at p of a Gaussian centered at p0. Standard deviation ssq. """
    v = p - p0
    return exp(-v@v/ssq)

def voxel_neighbourhood(volume_dimensions, pi):
    """ This is a generator that allows us to loop over voxels in the predefined volume
    neighborhood around pi """
    for i,j,k in np.ndindex((5,5,5)):
        yield pi[0]-2+i, pi[1]-2+j, pi[2]-2+k

class XForm:
    """ This class transforms grid indices to the corresponding location
    in a 3D bounding box and back"""
    def __init__(self,Plow,Phigh,dims):
        """ Initialize with the minimum coordinates corner Plow, maximum coordinates
        corner Phigh, and the voxel grid dimensions. """
        self.dims = array(dims)
        self.Plow = array(Plow)
        self.Phigh = array(Phigh)
        self.scaling = (self.Phigh - self.Plow) / self.dims
        self.inv_scaling = self.dims / (self.Phigh - self.Plow)
    def map(self,pi):
        """ map transforms a grid point to the point in space that corresponds to 
        the grid point. """
        return self.scaling * array(pi) + self.Plow
    def inv_map(self,p):
        return array((p-self.Plow) * self.inv_scaling,dtype=int)


### Part 2: Loading data and specifying a subset to use
We now load the point cloud and select a subset of points. Once your code is running, play around with the number of points that you load to get a better result in a reasonable time.

In [ ]:
# Specify how many points to use, experiment with this number <----------------------
NO_POINTS = 1000

# Below, we load the data set. 
print("Loading point set ...")
f = open("owl-lines.off")
coords = [list(map(float, l.split())) for l in f.readlines()]
shuffle(coords)
coords = array(coords)
point_mat = array(coords[0:NO_POINTS,0:3])
norm_mat = array(coords[0:NO_POINTS,3:6])

#Create and slightly expand bbox
corner_min = point_mat.min(0)
corner_max = point_mat.max(0)
print("bbox : ", corner_min, corner_max)
diag_vec = corner_max - corner_min
diag = norm(diag_vec)
corner_min -= 0.1 * diag_vec
corner_max += 0.1 * diag_vec

### Part 3: Volume dimensions
Here we compute the volume dimensions such that the longest axis is leng_axis. Afterwards we compute the scaling factor of the Gauss kernel. Once your code is up and running play around with these two values to achieve a better solution.

In [ ]:
# Once the code works, experiment with the dimensions of the volume. <----------------------
leng_axis=256.0
volume_dimensions = np.asarray((leng_axis/ diag_vec.max()) * diag_vec,dtype=int)
print("Volume dimension : ", volume_dimensions)

# Initialize the volume that will contain sum of distance values
volume = np.zeros(volume_dimensions,dtype=float)

# Initialize the weight volume
volume_w = np.zeros(volume_dimensions,dtype=float)

# Create a transform mapping between voxel grid and point bbox.
xform = XForm(corner_min,corner_max,volume.shape)

# Compute 'sigma square' (the scaling factor for the Gauss kernel) based on the length
# of the bbox diagonal. Experiment with this number <-------------------------
ssq = 2*5

### Part 4: The Actual Exercise
We need to compute a contribution to the volume for every point in the selected subset. This is done by iterating over all voxels in a neighborhood around a given point. Enter your code here.

In [ ]:
# Iterate over all points
# for each point compute its contribution to the local neighborhood.
for point in point_mat:
    # Map the point to the voxel grid
    pi = xform.inv_map(point)
    # Iterate over the neighborhood of voxels around the point
    for pi_n in voxel_neighbourhood(volume_dimensions, pi):
        # Check if the voxel is within bounds
        if np.all(np.array(pi_n) >= 0) and np.all(np.array(pi_n) < volume_dimensions):
            # Map the voxel back to space
            p_n = xform.map(pi_n)
            # Compute the Gaussian contribution
            g = Gaussian(point, p_n, ssq)
            # Update the volume and weight volume
            volume[pi_n] += g
            volume_w[pi_n] += 1

# Finally, normalize the volume
for i in range(volume.shape[0]):
    for j in range(volume.shape[1]):
        for k in range(volume.shape[2]):
            if volume_w[i, j, k] > 0:
                volume[i, j, k] /= volume_w[i, j, k]

### Part 5: Visualize
In this part, we polygonize and display the number vertices and faces, aswell as the boundary of the box.

In [ ]:
m = hmesh.volumetric_isocontour(volume, bbox_min=corner_min, bbox_max=corner_max)
jd.display(m)

### Part 7: Questions
- Why use volumetric reconstruction instead of Delaunay triangulation, say? \
We use volumetric reconstruction (vr) because it allows us to combine data and estimate a consistent surface even when the data is noisy or missing regions. Delaunay triangulations connects points geometrically but does not inherently distinguishthe objects surface from the interior or background. SO if we want to be able to distinquosh the object from the background, it would be smarter to use volumetric reconstruction.
- What is the purpose of the XForm class and how does it work? \ purpose of the class is to connect the 3D voxel grid to real world coordinates within a defined bounding box. It works by using the bounding box limits and grid dimensions to calculate scaling factors which allows the points to b converted between voxel indices and real-world coordinates both ways.
- What is your experience about the number of points you used? \
100 just gave small boxes distributed like a latter \
1000 more boxes and starting more of a shape \
10000 slowly starts to look like an owl but still not very good also slower to compute\
- What is your experience about the influence of setting the volume dimensions (leng_axis)? \
256/2 larger boxes (could compare it to a lower resolution) \
256*2 gives holes in the shape and is slower to compute, you need more points to make it fill out the complete figure than you with 256 or 128 \
- What is your experience about the Gauss kernel scaling factor (ssq)? \
I had a hard time telling the difference between the 3D plots with different ssq values.